# Benchmarking `Baseline` — the minimal example

Everything you need to benchmark the `Baseline` knowledge-graph pipeline, as briefly as possible.

The five stage benchmarks — `Dedup`, `Chunking`, `Extraction`, `Resolution`, `Quality` — each isolate ONE pipeline stage and score it against the bundled benchmarking gold data. Every run returns a `StageResult` — just `print()` it to see a readable per-pipeline report. (`RAG` needs its own gold QA corpus, so it's covered in the full tutorial instead.)

> **Only bundled benchmarking data is used** — `benchmarks/data/*_gold.jsonl` ships with the repo; the notebook never creates or authors a dataset. `max_records` merely caps how many bundled gold records a stage scores, so the demo stays fast.

> **If you just updated the library:** do **Kernel → Restart & Run All** — re-running cells alone keeps the old modules in memory. The library fails loudly (no silent fallbacks): if something breaks, it raises.


In [61]:
# ── Setup: imports + bundled benchmarking data ──────────────────
from pathlib import Path

from kglab._shared.stage_config import PreprocessConfig
from kglab.benchmark_pipeline import Benchmark
from kglab.pipelines import Baseline


def _find_root() -> Path:
    """Repo root = the directory holding pyproject.toml + src/kglab."""
    for cand in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (cand / "pyproject.toml").exists() and (cand / "src" / "kglab").is_dir():
            return cand
    raise RuntimeError("Could not find the kglab repo root above the working directory.")


REPO_ROOT = _find_root()

# Only the benchmarking data that ships with the repo is used — the notebook
# never creates or authors a dataset.
GOLD = REPO_ROOT / "benchmarks" / "data"  # *_gold.jsonl per stage

print("gold:", GOLD)

gold: /Users/victorpekkari/Desktop/aiv/kglab/benchmarks/data


In [62]:
# ── 1) Stage benchmarks, one readable report ────────────────────
# Each stage isolates ONE pipeline component and scores it against the bundled
# gold data. `max_records` caps how many bundled gold records a stage scores
# (a slice of the same file) so the demo stays fast while the metric stays honest.

# Dedup — near-duplicate detection against DBLP-ACM gold (full bundled gold).
dedup = Benchmark.Dedup(dataset=GOLD / "dedup_gold.jsonl").run(
    pipelines={
        "minhash": Baseline(preprocess=PreprocessConfig(doc_dedup_method="minhash")),
    }
)

# Chunking — do chunk boundaries keep gold entities intact? First 200 gold records.
chunking = Benchmark.Chunking(dataset=GOLD / "chunking_gold.jsonl").run(
    pipelines={
        "sentence": Baseline(preprocess=PreprocessConfig(chunk_method="sentence")),
    },
    max_records=200,
)

# Extraction — how well does spaCy NER match gold spans? First 500 gold records.
extraction = Benchmark.Extraction(dataset=GOLD / "ner_gold.jsonl").run(
    pipelines={
        "spacy": Baseline(),
    },
    max_records=500,
)

# Resolution — entity merging quality (cluster F1), full bundled gold.
resolution = Benchmark.Resolution(dataset=GOLD / "resolution_gold.jsonl").run(
    pipelines={
        "string": Baseline(),
    }
)

# Quality — keep/reject filter accuracy, full bundled gold.
quality = Benchmark.Quality(dataset=GOLD / "quality_gold.jsonl").run(
    pipelines={
        "surface": Baseline(),
    }
)

# Each stage returned a StageResult — print() it to see its report table
# (headline metric marked with '*', plus a plain-English interpretation).
print(dedup)
print(chunking)
print(extraction)
print(resolution)
print(quality)


Dedup — near-duplicate detection (DBLP-ACM gold)   [dataset: dedup_gold.jsonl]
pipeline | precision (TP/(TP+FP)) | recall (TP/(TP+FN)) | f1 (2·TP/(2·TP+FP+FN)) | accuracy ((TP+TN)/N) | threshold | runtime_s
---------+------------------------+---------------------+------------------------+----------------------+-----------+----------
minhash  |                  1.000 |               0.229 |                0.373 * |                0.615 |     0.850 |     3.270
  * = best f1
  TP = true positives · FP = false positives · FN = false negatives · TN = true negatives
  Higher F1 = better duplicate detection. Case-sensitive methods score low recall on DBLP-ACM (the duplicates differ in case) — an honest finding, not a bug.


Chunking — chunk boundaries keep gold entities intact   [dataset: chunking_gold.jsonl]
pipeline | precision (TP/(TP+FP)) | recall (TP/(TP+FN)) | f1 (2·TP/(2·TP+FP+FN)) | n_samples | runtime_s
---------+------------------------+---------------------+-----------------------

## Reading the numbers

All data is bundled with the repo. Where a stage shows `n_samples` below the full gold size, that's `max_records` scoring a slice of the same bundled file — the metric is still measured on real gold, just fewer records (keeps the demo fast; drop the cap for the full run).

Each stage isolates ONE pipeline stage against gold data, so a low score points at the weak stage:

- `Dedup` — near-duplicate detection (DBLP-ACM gold). Case-sensitive methods score low *recall* — an honest finding, not a bug.
- `Chunking` / `Extraction` — whether chunks keep gold entities intact / how well spaCy NER matches gold spans. The `spacy` pipeline is `en_core_web_lg` (the spaCy extractor's default); pass `entity_options={"model_name": "en_core_web_sm"}` to compare a smaller model.
- `Resolution` / `Quality` — **smoke tests, not real measurements**: the bundled golds are tiny placeholders (2 hand-written clusters / 9 hand-written records), so a 1.0 or 0.0 here carries no meaning. Real gold (T2D / TACRED) is license-gated — supply it via `dataset=` or `--dataset`.

> **Extraction has several golds — pick the one that fits your extractor** (`print(Benchmark.Extraction.golds())`): CoNLL (train/test — PER/LOC/ORG/MISC), `wikiann` (PER/ORG/LOC), `FewNERD` (fine-grained). Fetch one with `Data.download("bench_ner_test", path=...)` and pass it via `dataset=`. **Prefer test splits**: the bundled `ner_gold.jsonl` is the CoNLL *train* split — not a held-out evaluation (models may have seen it), so the demo numbers above are indicative, not final.

`RAG` is not shown here: it needs a gold QA corpus of its own (the bundled HotpotQA gold is network/license-gated). See the full tutorial to run it with your own gold.

Swap `Baseline` for `Semantic()` (or any custom `Pipeline` subclass) in the cells above to compare pipelines.
